# Step 4: Model Training

Train a **RandomForest classifier** using pre-computed features and register in Snowflake Model Registry.

## Features Used

The model is trained on **raw features + computed features** from preprocessing:
- **Raw**: Age, BMI, Heart Rate, Blood Pressure, Lab Values, etc.
- **Computed**: SHOCK_INDEX, PULSE_PRESSURE, BMI_CATEGORY, VITAL_SIGNS_SEVERITY

## Snowflake Services Used

| Service | Purpose |
|---------|---------|
| **ML Jobs** | Remote training on SPCS compute pools |
| **Model Registry** | Version and store models |

## Prerequisites

- Run notebooks 01-03 first (03 creates TRAINING_FEATURES and TEST_FEATURES tables)

## Imports and Configuration

In [ ]:
%cd ..
%load_ext autoreload

In [ ]:
import json
import os
import sys
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

from snowflake.snowpark import Session
from source.configs import get_config, config_to_dict
from source.utils import get_session, get_model_version
from source.framework.train import RemoteTrainer

config = get_config("source/config.yaml")
session = get_session(config.snowflake.connection_name)

DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
COMPUTE_WAREHOUSE = config.snowflake.warehouse

session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(COMPUTE_WAREHOUSE)

print(f"Connected as: {session.get_current_user()}")
print(f"Current role: {session.get_current_role()}")
print(f"Current warehouse: {session.get_current_warehouse()}")

In [ ]:
COMPUTE_POOL = config.compute.compute_pool

pool_exists = session.sql(f"SHOW COMPUTE POOLS LIKE '{COMPUTE_POOL}'").collect()
if not pool_exists:
    session.sql(f"""
        CREATE COMPUTE POOL {COMPUTE_POOL}
        MIN_NODES = 1 MAX_NODES = 1
        INSTANCE_FAMILY = CPU_X64_S
        AUTO_SUSPEND_SECS = 300
    """).collect()
    print(f"Created compute pool: {COMPUTE_POOL}")
else:
    print(f"Compute pool exists: {COMPUTE_POOL}")

# ML Jobs: Remote Model Training

In [ ]:
COMPUTE_POOL = config.compute.compute_pool
stage = f"{DB}.{SCHEMA}.{config.stages.job_payloads}"
num_instances = config.train.num_nodes
dataset_name = config.feature_store.training_dataset_name

trainer = RemoteTrainer(
    session=session,
    compute_pool=COMPUTE_POOL,
    stage=stage,
    source_dir="source",
)

job = trainer.submit(
    entrypoint="train.py",
    num_instances=num_instances,
    env_vars={
        "ML_PIPELINE_CONFIG":        json.dumps(config_to_dict(config)),
        "TRAINING_DATASET_NAME":     dataset_name,
        "TRAINING_DATASET_VERSION":  dataset_version or "",
    },
)
print(f"Job submitted: {job.id}")
print(f"Status: {job.status}")
print(f"Distributed training: {num_instances} node(s) on compute pool '{COMPUTE_POOL}'")
print(f"Training dataset: {dataset_name} / {dataset_version}")

In [ ]:
trainer.wait_and_log(job)

## Retrieve Registered Model Version

In [ ]:
model_name = config.model.model_name
latest_version = get_model_version(session, DB, SCHEMA, model_name)
version_name = latest_version.version_name

print(f"Registered version: {model_name}/{version_name}")

## Model Evaluation & Promotion Gate

In [ ]:
from source.framework.evaluator import Evaluator
from source.utils import get_feature_config

test_table = f"{DB}.{SCHEMA}.{config.tables.test_features}"
metrics_table = f"{DB}.{SCHEMA}.{config.tables.metrics_table}"
promotion_thresholds = {
    "accuracy": config.evaluation.accuracy_threshold,
    "f1_macro":  config.evaluation.f1_macro_threshold,
}

feature_config = get_feature_config(config)
feature_columns = [c.upper() for c in feature_config["all_numeric_features"] + feature_config["all_categorical_features"]]
target_column = feature_config["target_column"].upper()
class_labels = feature_config["class_labels"]

evaluator = Evaluator(session=session)
metrics = evaluator.evaluate_from_registry(
    model_name=model_name,
    model_version=version_name,
    registry_database=DB,
    registry_schema=SCHEMA,
    test_table=test_table,
    feature_columns=feature_columns,
    target_column=target_column,
    class_labels=class_labels,
)

print(f"accuracy:  {metrics['accuracy']:.4f}")
print(f"f1_macro:  {metrics['f1_macro']:.4f}")
print(f"test_size: {metrics['test_size']}")

In [ ]:
evaluator.log_metrics(
    metrics=metrics,
    metrics_table=metrics_table,
    model_name=model_name,
    model_version=version_name,
)

mv = latest_version
scalar_metrics = {k: v for k, v in metrics.items() if isinstance(v, (int, float))}
for metric_name, metric_value in scalar_metrics.items():
    try:
        mv.log_metric(metric_name, metric_value)
    except Exception as e:
        logger.warning("Could not log metric '%s' to registry: %s", metric_name, e)

print(f"Logged {len(scalar_metrics)} metrics to {metrics_table} and model version {version_name}")

In [ ]:
promotion_result = evaluator.check_promotion_criteria(
    metrics=metrics,
    thresholds=promotion_thresholds,
)

report = evaluator.generate_report(metrics, promotion_result)
print(report)